# 05 · Assemble it: a tiny AI-shaped workflow
**⏱ 12 minutes · Python concepts working together in one application**

It's time to stop studying syntax in isolation. We'll follow a support ticket from input validation to a proposed response, through simulated asynchronous tools and an approval checkpoint.

**Important:** this is a **deterministic agent-shaped workflow**, not an autonomous LLM agent. We start without a model so we can inspect the Python mechanics. The [optional Gradio + Ollama showcase](../guides/06_local_ai_lab.md) adds a real local model afterward.

By the end you should understand *where* an actual model can be inserted, and why we keep safety and validation in ordinary application code.

## 1. Recognize the architecture from Java

If you were designing this in Spring Boot, you might have:

```text
TicketRequest DTO
    ↓ validate
TicketService
    ├─ CategoryClassifier
    ├─ KnowledgeBaseClient
    └─ CustomerContextClient
    ↓ combine
ReplyProposal DTO
    ↓ approval required
(External action — intentionally NOT implemented)
```

Our Python project follows the same separation, with very little boilerplate:

- `models.py`: validated Pydantic data contracts.
- `tools.py`: deterministic classification and two fake async tools.
- `workflow.py`: concurrent orchestration and **pure approval simulation**.

**Why separate them?** So you can test the workflow without paying for a model, without contacting a network, and without accidentally sending a real message.

## 2. Validate before any work happens

A support ticket is input from outside the program. Pydantic rejects bad values *before* classification. In Java this is like validating a request DTO at your controller boundary.

**Pause:** will the ticket below pass the contract?

In [ ]:
from mini_agent import Ticket

ticket = Ticket(
    id=101,
    subject="Cannot sign in",
    body="I can't log in to my account after changing the password.",
)
print("Validated request:", ticket.model_dump())
assert ticket.id == 101

## 3. Call the asynchronous workflow

`prepare_reply()` classifies the request, then awaits two simulated tools **concurrently** with `asyncio.gather`. It returns a structured `ReplyProposal` whose `approved` field defaults to `False`.

This is a natural extension of Lesson 03: independent I/O can overlap, but the program still controls which actions can happen.

**Predict:** what category will be chosen?

In [ ]:
from mini_agent import prepare_reply

proposal = await prepare_reply(ticket)
print("Proposed reply:", proposal.model_dump())
assert proposal.category == "access"
assert proposal.approved is False

## 4. Propose first; act only with explicit approval

Tool-calling AI systems shouldn't equate *the model suggested an action* with *the application is allowed to execute it*. Think of a Java service that builds a command and waits for a separate authorization decision.

Our function is intentionally called `simulate_send`, and it returns a hypothetical `would_send` flag. **Even with approval set to True, the demo never sends anything.**

Run both cases and inspect the results.

In [ ]:
from mini_agent import simulate_send

rejected = simulate_send(proposal, approved=False)
approved = simulate_send(proposal, approved=True)
print("Not approved:", rejected)
print("Hypothetically approved:", approved)
assert rejected["would_send"] is False
assert approved["would_send"] is True

## 5. Change the input: does the same pipeline still work?

An application becomes interesting when it handles different cases. Try a **billing question** and an unrelated **feature request** without changing the workflow code.

**Before running:** predict the category of both. If your prediction is wrong, inspect `src/mini_agent/tools.py` and see the keyword rules used for classification.

In [ ]:
billing = Ticket(id=102, subject="Invoice question", body="Where can I download my invoice?")
feature = Ticket(id=103, subject="Suggestion", body="Please add a dark theme")

billing_reply = await prepare_reply(billing)
feature_reply = await prepare_reply(feature)
print("Billing:", billing_reply.category, billing_reply.suggested_reply)
print("Feature:", feature_reply.category, feature_reply.suggested_reply)
assert billing_reply.category == "billing"
assert feature_reply.category == "general"

## 6. Look inside the implementation

Don't treat this as a mysterious black box. Python's `inspect` module lets you examine a function signature, and GitHub/Cursor/VS Code can open the actual source.

Take a minute to inspect:

- [Typed input/output models](../src/mini_agent/models.py)
- [Read-only simulated tools](../src/mini_agent/tools.py)
- [Orchestration and approval gate](../src/mini_agent/workflow.py)

Ask: **which parts would stay exactly the same if we replaced the keyword classifier with a local language model?**

In [ ]:
import inspect
from mini_agent import prepare_reply

print("Workflow signature:", inspect.signature(prepare_reply))
print("Async function:", inspect.iscoroutinefunction(prepare_reply))
assert inspect.iscoroutinefunction(prepare_reply)

### Your small change (2 minutes)

Write a new `Ticket` for a password-reset issue and verify it classifies as `access`. Then consider one improvement that **doesn't require AI**: clearer validation, a new category, or stronger tests.

The exercise below includes a valid starting example. Replace the subject and body with your own wording.

In [ ]:
my_ticket = Ticket(id=104, subject="Password reset", body="I cannot reset the password for my account")
my_reply = await prepare_reply(my_ticket)
print(my_reply.category, my_reply.suggested_reply)
assert my_reply.category == "access"

### Wrap-up: where Python matters for AI engineering

You've now used **Pydantic models**, package imports, `async` orchestration, fake I/O, dictionary-based decisions, typed results and a human-approval boundary in one small system.

That's the language bridge. You didn't need to build a production-scale agent to see the essential Python patterns.

**Checkpoint:** why is `prepare_reply()` allowed to generate a proposal without permission to send it? What extra controls would a real application need (authorization, audit, idempotency, human approval)?

**Next:** [06 · Readiness check](06_readiness_check.ipynb). Want a browser UI and a *real local model*? See the [optional Gradio/Ollama lab](../guides/06_local_ai_lab.md).